<a href="https://colab.research.google.com/github/rifahnanjiba02-arch/vision-language-composition/blob/main/scripts/MACCO_Flickr30k_Retrieval_CIFAR_ZeroShot_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MACCO — Flickr30k Retrieval and CIFAR Zero-Shot Classification

**Four frozen models, no training.** We measure Flickr30k image–text retrieval Recall@1/@5/@10 and CIFAR-10/CIFAR-100 zero-shot Top-1/Top-5 classification accuracy.

**Dataset:** [aplominski/flickr30k-multilingual](https://huggingface.co/datasets/aplominski/flickr30k-multilingual), **test** split only: **1,000 images** × **5 original English captions** = **5,000 captions**. This dataset republishes Flickr30k with Karpathy train/validation/test splits. We use only `image` and `captions_en`, not translations.

**Run:** GPU runtime → execute the code cells in order. The retrieval and classification sections each save a CSV. Every model uses the same clean test examples and preprocessing within each evaluation. Models are **not trained**.


In [1]:
%pip -q install open_clip_torch datasets huggingface_hub pandas tqdm


In [2]:
# 1. Imports + exact four checkpoints
import gc
import torch
import torch.nn.functional as F
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset
from huggingface_hub import hf_hub_download
import open_clip

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
IMAGE_BATCH = 32 if DEVICE == 'cuda' else 8
TEXT_BATCH = 256 if DEVICE == 'cuda' else 32

MODELS = {
    'Author MACCO': ('hiker-lw/MACCO', 'MACCO-CLIP-ViT-B-32.pt'),
    'Our W1+CML': ('rifahnanjiba2/MACCO-Finetuned', 'MACCO-W1-CML-ViT-B-32.pt'),
    'Our Pixel Noise': ('rifahnanjiba2/MACCO-Finetuned', 'noise_adaption/input_pixel_noise/MACCO-W1-CML-InputPixelNoise-ViT-B-32.pt'),
    'Our Traffic': ('rifahnanjiba2/MACCO-Finetuned', 'traffic_adaptation/clip_only/MACCO-W1-CML-TrafficMaskNoise-ViT-B-32.pt'),
}
print('Device:', DEVICE)
print('Models:', list(MODELS))


Device: cuda
Models: ['Author MACCO', 'Our W1+CML', 'Our Pixel Noise', 'Our Traffic']


In [3]:
# 2. Download ONLY Flickr30k TEST images and English captions
DATASET = 'aplominski/flickr30k-multilingual'
ds = load_dataset(DATASET, split='test')
assert len(ds) == 1000 and {'image', 'captions_en'}.issubset(ds.column_names)

caption_groups = ds['captions_en']
assert all(len(group) == 5 and all(isinstance(x,str) and x.strip() for x in group) for group in caption_groups)
captions = [caption for group in caption_groups for caption in group]
caption_image_ids = torch.arange(len(ds)).repeat_interleave(5)
print(f'Ready: {len(ds)} test images, {len(captions)} English captions')
print('First caption:', captions[0])


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:121: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


README.md:   0%|          | 0.00/3.15k [00:00<?, ?B/s]

data/train-00000-of-00008.parquet: reconstructing file:   0%|          |  0.00B /  501MB            

data/train-00000-of-00008.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00008.parquet: reconstructing file:   0%|          |  0.00B /  512MB            

data/train-00001-of-00008.parquet: downloading bytes:           |  0.00B            

... (16 outputs left out)

Generating train split:   0%|          | 0/29769 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1014 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Ready: 1000 test images, 5000 English captions
First caption: The man with pierced ears is wearing glasses and an orange hat .


In [6]:
# 3. Encode images and captions, then measure retrieval Recall@K
@torch.inference_mode()
def test_one_model(repo_id, filename):
    checkpoint = hf_hub_download(repo_id=repo_id, filename=filename)
    model, _, preprocess = open_clip.create_model_and_transforms(
        'ViT-B-32', pretrained=checkpoint, device=DEVICE, precision='fp32'
    )
    model.eval()

    image_vectors = []
    for start in tqdm(range(0, len(ds), IMAGE_BATCH), desc='Images', leave=False):
        batch = torch.stack([
            preprocess(ds[i]['image'].convert('RGB'))
            for i in range(start, min(start + IMAGE_BATCH, len(ds)))
        ]).to(DEVICE)
        image_vectors.append(F.normalize(model.encode_image(batch).float(), dim=-1).cpu())

    text_vectors = []
    for start in tqdm(range(0, len(captions), TEXT_BATCH), desc='Captions', leave=False):
        tokens = open_clip.tokenize(captions[start:start + TEXT_BATCH]).to(DEVICE)
        text_vectors.append(F.normalize(model.encode_text(tokens).float(), dim=-1).cpu())

    # 1,000 images × 5,000 captions. The same galleries are used for every checkpoint.
    similarity = torch.cat(image_vectors) @ torch.cat(text_vectors).T
    i2t_top10 = similarity.topk(10, dim=1).indices
    t2i_top10 = similarity.T.topk(10, dim=1).indices

    # Any of an image's 5 captions is a valid image→text retrieval result.
    i2t_hit = caption_image_ids[i2t_top10].eq(torch.arange(len(ds)).unsqueeze(1))
    # Every caption has exactly one matching image in text→image retrieval.
    t2i_hit = t2i_top10.eq(caption_image_ids.unsqueeze(1))

    scores = {}
    for k in (1, 5, 10):
        scores[f'I2T_R@{k}'] = round(100 * i2t_hit[:, :k].any(dim=1).float().mean().item(), 2)
        scores[f'T2I_R@{k}'] = round(100 * t2i_hit[:, :k].any(dim=1).float().mean().item(), 2)

    del model, image_vectors, text_vectors, similarity
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    return scores


In [7]:
# 4. Run all models; print results and export one CSV
rows = []
for name, (repo, filename) in MODELS.items():
    print(f'\nEvaluating: {name}')
    scores = test_one_model(repo, filename)
    rows.append({'Model': name, **scores})
    result = pd.DataFrame(rows)
    result.to_csv('MACCO_Flickr30k_Recall.csv', index=False)  # also saves partial progress
    display(result.tail(1))

print('\nFinal recall comparison (%) — higher is better')
display(result)
print('Saved: MACCO_Flickr30k_Recall.csv')

# Optional Colab download (comment out if you prefer to leave the file in /content).
try:
    from google.colab import files
    files.download('MACCO_Flickr30k_Recall.csv')
except ImportError:
    pass



Evaluating: Author MACCO


Images:   0%|          | 0/32 [00:00<?, ?it/s]

Captions:   0%|          | 0/20 [00:00<?, ?it/s]

,Model,I2T_R@1,T2I_R@1,I2T_R@5,T2I_R@5,I2T_R@10,T2I_R@10
0,Author MACCO,79.3,65.2,94.2,87.2,97.3,92.62



Evaluating: Our W1+CML


MACCO-W1-CML-ViT-B-32.pt: reconstructing file:   0%|          |  0.00B /  605MB            

... (17 outputs left out)

,Model,I2T_R@1,T2I_R@1,I2T_R@5,T2I_R@5,I2T_R@10,T2I_R@10
0,Author MACCO,79.3,65.20,94.2,87.20,97.3,92.62
1,Our W1+CML,79.3,65.64,94.4,87.64,97.4,93.10
2,Our Pixel Noise,79.9,65.70,94.8,87.76,97.7,93.00
3,Our Traffic,79.3,65.58,94.0,87.60,97.3,93.06


Saved: MACCO_Flickr30k_Recall.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Download CIFAR-10 and CIFAR-100 test data
We'll use 1,000 balanced test images from each dataset.

In [8]:
from torchvision.datasets import CIFAR10, CIFAR100
import random

SEED = 42

datasets = {
    "CIFAR-10": CIFAR10(root="/content/data", train=False, download=True),
    "CIFAR-100": CIFAR100(root="/content/data", train=False, download=True),
}

def balanced_sample(dataset, n_per_class):
    rng = random.Random(SEED)
    indices = []
    for label in range(len(dataset.classes)):
        candidates = [
            i for i, y in enumerate(dataset.targets) if y == label
        ]
        indices.extend(rng.sample(candidates, n_per_class))
    return sorted(indices)

test_indices = {
    "CIFAR-10": balanced_sample(datasets["CIFAR-10"], 100),
    "CIFAR-100": balanced_sample(datasets["CIFAR-100"], 10),
}

for name, indices in test_indices.items():
    print(f"{name}: {len(indices)} test images")

100%|██████████| 170M/170M [29:48<00:00, 95.3kB/s]
100%|██████████| 169M/169M [28:19<00:00, 99.4kB/s]


CIFAR-10: 1000 test images
CIFAR-100: 1000 test images


#  Zero-shot classification function

In [9]:
@torch.inference_mode()
def zero_shot_accuracy(model, preprocess, dataset, indices):
    model.eval()

    prompts = [
        f"a photo of a {name.replace('_', ' ')}."
        for name in dataset.classes
    ]

    tokens = open_clip.tokenize(prompts).to(DEVICE)
    text_features = F.normalize(
        model.encode_text(tokens).float(), dim=-1
    )

    correct1, correct5 = 0, 0
    batch_size = 32 if DEVICE == "cuda" else 8

    for start in tqdm(range(0, len(indices), batch_size)):
        batch_indices = indices[start:start + batch_size]

        images = torch.stack([
            preprocess(dataset[i][0].convert("RGB"))
            for i in batch_indices
        ]).to(DEVICE)

        labels = torch.tensor(
            [dataset.targets[i] for i in batch_indices],
            device=DEVICE
        )

        image_features = F.normalize(
            model.encode_image(images).float(), dim=-1
        )

        similarity = image_features @ text_features.T
        top5 = similarity.topk(5, dim=1).indices

        correct1 += (top5[:, 0] == labels).sum().item()
        correct5 += (top5 == labels[:, None]).any(dim=1).sum().item()

    n = len(indices)
    return {
        "Top-1": round(100 * correct1 / n, 2),
        "Top-5": round(100 * correct5 / n, 2)
    }

# Evaluate all four models and save CSV

In [10]:
zero_shot_rows = []

for name, (repo, filename) in MODELS.items():
    print(f"\nEvaluating: {name}")

    checkpoint = hf_hub_download(
        repo_id=repo,
        filename=filename
    )

    model, _, preprocess = open_clip.create_model_and_transforms(
        "ViT-B-32",
        pretrained=checkpoint,
        device=DEVICE,
        precision="fp32"
    )
    model.eval()

    row = {"Model": name}

    for dataset_name, dataset in datasets.items():
        print(f"Dataset: {dataset_name}")

        scores = zero_shot_accuracy(
            model,
            preprocess,
            dataset,
            test_indices[dataset_name]
        )

        row[f"{dataset_name} Top-1"] = scores["Top-1"]
        row[f"{dataset_name} Top-5"] = scores["Top-5"]

    zero_shot_rows.append(row)

    zero_shot_results = pd.DataFrame(zero_shot_rows)
    zero_shot_results.to_csv(
        "MACCO_ZeroShot_Classification.csv",
        index=False
    )

    display(zero_shot_results.tail(1))

    del model
    gc.collect()

    if DEVICE == "cuda":
        torch.cuda.empty_cache()

print("\nFinal Zero-Shot Classification Results (%)")
display(zero_shot_results)

from google.colab import files
files.download("MACCO_ZeroShot_Classification.csv")


Evaluating: Author MACCO
Dataset: CIFAR-10


  0%|          | 0/32 [00:00<?, ?it/s]

Dataset: CIFAR-100


  0%|          | 0/32 [00:00<?, ?it/s]

,Model,CIFAR-10 Top-1,CIFAR-10 Top-5,CIFAR-100 Top-1,CIFAR-100 Top-5
0,Author MACCO,85.3,99.8,63.3,87.2



Evaluating: Our W1+CML
Dataset: CIFAR-10


... (14 outputs left out)


Final Zero-Shot Classification Results (%)


,Model,CIFAR-10 Top-1,CIFAR-10 Top-5,CIFAR-100 Top-1,CIFAR-100 Top-5
0,Author MACCO,85.3,99.8,63.3,87.2
1,Our W1+CML,85.7,99.7,63.4,87.6
2,Our Pixel Noise,86.0,99.7,63.6,87.8
3,Our Traffic,85.8,99.7,63.7,87.4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>